# parameter-wrap-around-tensor — worked example 2: two-step Parameter(Tensor(array)) wrap

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `parameter-wrap-around-tensor`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Weights are built as `Parameter(Tensor(np.random...))`: an ndarray is wrapped in a Tensor, which is then flagged as a Parameter. The result is simultaneously ndarray-backed, Tensor-typed, and Parameter-tagged, so it passes every isinstance gate.

## Worked solution

We define `MiniTensor` (wraps an ndarray) and `Parameter(MiniTensor)` (the trainable tag). The idiom builds a weight in two conceptual steps: create the array, wrap it as a MiniTensor, and flag it as a Parameter. Because Parameter is a MiniTensor subclass, the single `Parameter(array)` call gives us all three properties at once. We seed numpy, build a weight this way, and print that it is both a MiniTensor and a Parameter, that its backing data is an ndarray, and that `requires_grad` is True.

In [ ]:
import numpy as np

np.random.seed(0)

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

weight = Parameter(np.random.randn(2, 3))
print('is MiniTensor:', isinstance(weight, MiniTensor))
print('is Parameter:', isinstance(weight, Parameter))
print('ndarray-backed:', isinstance(weight.array, np.ndarray))
print('requires_grad:', weight.requires_grad)